# Loading the datasets

In [38]:
#Importar librerias
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import ptitprince as pt
pd.options.display.max_rows = 999
import warnings
warnings.filterwarnings("ignore")
import sys
sys.dont_write_bytecode = True

In [113]:
# Replace
path = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5\Data Tables\HDeviceCGM.txt"
def Loading_File(file_path):
    MasterDF = pd.read_csv(file_path, sep='|', header=0, low_memory = False)
    MasterDF = MasterDF[MasterDF['RecordType'] == 'CGM']
    #Creating a date time column
    MasterDF['Today'] = datetime.today().date()
    MasterDF['Date'] = MasterDF['Today'] + pd.to_timedelta(MasterDF['DeviceDtTmDaysFromEnroll'], unit='d')
    MasterDF['DeviceTm'] = MasterDF.DeviceTm.astype('str')
    MasterDF['DeviceTm'] = MasterDF['DeviceTm'].str[:-2]+ '00'
    MasterDF['DateTime'] = MasterDF.Date.astype('str')+ ' '+ MasterDF.DeviceTm
    MasterDF['DateTime'] = pd.to_datetime(MasterDF.DateTime, format='%Y-%m-%d %H:%M:%S')
    #selecting just the columns for Giammarino's code to run
    MasterDF = MasterDF[['PtID','DateTime','GlucoseValue']]
    MasterDF = MasterDF.rename(columns={'DateTime':'ts','PtID':'id','GlucoseValue':'gl'})
    MasterDF= MasterDF.reset_index(drop=True)
    MasterDF = MasterDF.drop_duplicates(subset=['ts','id'])
    # data = MasterDF.pivot(index='ts', columns=['id'], values=['gl'])
    # data.columns = data.columns.get_level_values(level='id')
    return MasterDF
Replace = Loading_File(path)

In [ ]:
# AIDE
path = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\AIDE T1D\Data Tables\AIDEDeviceCGM.txt"
def Loading_File(file_path):
    MasterDF = pd.read_csv(file_path, sep='|', header=0, low_memory = False)
    MasterDF = MasterDF[MasterDF['RecordType'] == 'CGM']
    #Creating a date time column
    MasterDF['DateTime'] = MasterDF['DataDtTm']
    MasterDF['DateTime'] = pd.to_datetime(MasterDF.DateTime, errors='coerce', infer_datetime_format=True)
    #selecting just the columns for Giammarino's code to run
    MasterDF = MasterDF[['PtID','DateTime','GlucValue']]
    MasterDF = MasterDF.rename(columns={'DateTime':'ts','PtID':'id','GlucValue':'gl'})
    MasterDF= MasterDF.reset_index(drop=True)
    MasterDF = MasterDF.drop_duplicates(subset=['ts','id'])
    data = MasterDF.pivot(index='ts', columns=['id'], values=['gl'])
    data.columns = data.columns.get_level_values(level='id')
    return data
AIDE = Loading_File(path)

In [ ]:
# Shanghai
from pathlib import Path
T1D = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\Dataset Shangai\Shanghai_T1DM"
T2D = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\Dataset Shangai\Shanghai_T2DM"
def Loading_File(file_path):
    files = list(Path(file_path).glob('*.xls*'))
    folder_dfs = pd.DataFrame()
    # print(files[0])
    for file in files:
        try:
            pti = str(file)[124:128]
            df_temp = pd.read_excel(str(file))
            df_temp = df_temp.iloc[:,0:2]
            df_temp = df_temp.rename(columns={df_temp.columns[0]: 'DateTime'})
            df_temp = df_temp.rename(columns={df_temp.columns[1]: 'GlucValue'})
            df_temp['PtID'] = pti
            df_temp['DateTime'] = pd.to_datetime(df_temp.DateTime, errors='coerce', infer_datetime_format=True)
            df_temp = df_temp[['PtID','DateTime','GlucValue']]
            df_temp = df_temp.rename(columns={'DateTime':'ts','PtID':'id','GlucValue':'gl'})
            df_temp= df_temp.reset_index(drop=True)
            df_temp = df_temp.drop_duplicates(subset=['ts','id'])
        except Exception as e:
            print(f"Error al leer {file.name}: {e}")
        folder_dfs = pd.concat([folder_dfs, df_temp])
    return folder_dfs
Shanghai_T1D = Loading_File(T1D)
Shanghai_T2D = Loading_File(T2D)
Shanghai = pd.concat([Shanghai_T1D, Shanghai_T2D])
#pivoting the dataset
data = Shanghai.pivot(index='ts', columns=['id'], values=['gl'])
data.columns = data.columns.get_level_values(level='id')
Shanghai = data


# Transform them into Sequences

In [ ]:
from Datasets import Replace
from Datasets import AIDE
from Datasets import Shanghai


In [132]:
from src.New_Utils import New_Sequences
seq = New_Sequences(Replace)

In [133]:
len(seq)

0

In [127]:
Replace.head(9)

,id,ts,gl
0,183,2026-01-26 05:35:00,162.0
1,183,2026-01-26 05:30:00,164.0
2,183,2026-01-26 05:25:00,168.0
3,183,2026-01-26 05:20:00,169.0
4,183,2026-01-26 05:15:00,170.0
5,183,2026-01-26 05:10:00,171.0
6,183,2026-01-26 05:05:00,172.0
7,183,2026-01-26 05:00:00,174.0
8,183,2026-01-26 04:55:00,177.0


In [128]:
def Event(Data,glucose_threshold):
    'Data has to be type list'
    'Threshold should be an integer'
    C1 = 0
    C2 = 0
    for i in range(len(Data)) :
        if Data[i] < glucose_threshold:
            C1+=1
        if C1 == 3:
            C2+=1
        else:
            C1 = 0
        return 1 if C2 > 1 else 0

In [147]:
sequences = []
NotWorking = []
minutes = 5
Days_Week = 7 # this is the number of days to be considered in a week, can be changed
glucose_threshold = 54
    # calculate the number of timestamps in one week
sequence_length = int(Days_Week * 24 * 60 // minutes)
time_worn = 0.7

In [148]:
import math
data = Replace

for patient in data.id.unique():
    # Data = pd.DataFrame()
    #Do preprocessing inside the function per patient
    Data = data[data['id'].isin([patient])]
    Data = Data[Data['gl'].between(40, 400)]
    # reshape the dataset from long to wide
    Data = Data.pivot(index='ts', columns=['id'], values=['gl'])
    Data.columns = Data.columns.get_level_values(level='id')
    Data.reset_index(inplace = True)
    Data['date'] = pd.to_datetime(Data['ts']).dt.date
    
    Data=Data[Data[patient].notnull()]
    #Per Patient
    min_Date = datetime.strptime(str(Data['ts'].min())[:-9], '%Y-%m-%d').date()
    # print(min_Date)
    max_Date = datetime.strptime(str(Data['ts'].max())[:-9], '%Y-%m-%d').date()
    # print(max_Date)
    Difference = abs(max_Date-min_Date).days #difference in days between the two dates  
    Sequences = math.ceil(Difference/Days_Week) # Define the number of 1 week sequences
    Result = pd.DataFrame()
    for i in range (1, Sequences, 1):
        # generate the range
        date_generated = pd.DataFrame()    
        date_generated = [min_Date + timedelta(days=x) for x in range(0, (timedelta(days=Days_Week)).days)]
        # print(len(date_generated))
        min_Date = min_Date + timedelta(days=Days_Week)
        df_Generated = pd.DataFrame(date_generated)
        df_Generated = df_Generated.rename(columns={0:'date'})
        df_Generated = pd.merge(df_Generated,Data,'left',left_on='date',right_on='date')
        df_Generated['Sequence'] = i
        Result = pd.concat([Result,df_Generated])
    Grouped = Result.groupby('Sequence').count()
    Grouped['Total_Readings_Sequence'] = sequence_length
    Grouped = Grouped[['ts','Total_Readings_Sequence']]
    Grouped['Time_Worn'] = Grouped['ts']/Grouped['Total_Readings_Sequence']
    Grouped['Criteria'] = np.where(Grouped['Time_Worn']>time_worn, "Applicable", 'Not Applicable')
    Grouped = Grouped[Grouped['Criteria']=='Applicable']
    Result = Result[Result['Sequence'].isin(Grouped.index.to_series())]          
    for i in Grouped.index.to_series():
        #Evaluate if next is applicable
        try:
            if Grouped.loc[i+1]['Criteria'] == 'Applicable':
                X = Result[patient][Result['Sequence'] == i].dropna().to_list()
                L = len(X)
                Y = Result[patient][Result['Sequence'] == i+1].to_list()
                Y = Event(Y, glucose_threshold)
                #I need to determine the amount of consecutive data below the threshold and if it greater that 15 minutes (three readings) then Y = 1
                # save the patient's data
                sequences.append({
                'patient': patient,
                'Sequence': i,
                'start': str(Result['ts'][Result['Sequence'] == i].min()),
                'end': str(Result['ts'][Result['Sequence'] == i].max()),
                'L': L,
                'X': X,
                'Y': Y
                })
            else:
                pass
        except:
            pass
    else:
        pass


KeyboardInterrupt: 

In [149]:
import pandas as pd
import numpy as np

def New_Sequences_Fast(data, glucose_threshold=54, days_week=7, minutes=5):
    sequences = []
    # Pre-calculamos el largo esperado una sola vez
    expected_readings = int(days_week * 24 * 60 // minutes)
    
    # Aseguramos formatos fuera del bucle para no repetir trabajo
    data = data[(data['gl'] >= 40) & (data['gl'] <= 400)].copy()
    data['ts'] = pd.to_datetime(data['ts'])
    data['date'] = data['ts'].dt.date

    for patient, p_data in data.groupby('id'):
        # Ordenamos por tiempo para que diff() funcione
        p_data = p_data.sort_values('ts')
        
        min_date = p_data['date'].min()
        max_date = p_data['date'].max()
        
        # Generamos los cortes de cada 7 días
        current_start = min_date
        temp_sequences = []

        while current_start + pd.Timedelta(days=days_week) <= max_date:
            current_end = current_start + pd.Timedelta(days=days_week)
            next_end = current_end + pd.Timedelta(days=days_week)
            
            # 1. Semana Actual (X)
            mask_x = (p_data['date'] >= current_start) & (p_data['date'] < current_end)
            week_x = p_data[mask_x]
            
            # 2. Semana Siguiente (Y) - para la etiqueta
            mask_y = (p_data['date'] >= current_end) & (p_data['date'] < next_end)
            week_y = p_data[mask_y]
            
            # CRITERIO DE CALIDAD: 70% de datos
            if len(week_x) >= (expected_readings * 0.7) and len(week_y) >= (expected_readings * 0.7):
                
                # Usamos tu función Event sobre la semana siguiente
                # Convertimos a lista solo al final para ahorrar tiempo
                y_label = Event(week_y['gl'].tolist(), glucose_threshold)
                
                sequences.append({
                    'patient': patient,
                    'start': week_x['ts'].min(),
                    'end': week_x['ts'].max(),
                    'L': len(week_x),
                    'X': week_x['gl'].tolist(),
                    'Y': y_label
                })
            
            # Avanzamos la ventana (aquí decides si quieres solapamiento o no)
            current_start += pd.Timedelta(days=1) # Ventana deslizante de 1 día para más datos
            
    return sequences

In [ ]:
New_Sequences_Fast(Replace)

In [ ]:
import pandas as pd
import numpy as np

def New_Sequences_Fixed(data, glucose_threshold=54, days_week=7, minutes=5):
    sequences = []
    # 2016 lecturas para 7 días si es cada 5 min
    expected_readings = int(days_week * 24 * 60 // minutes)
    
    # Pre-procesamiento global
    data = data[(data['gl'] >= 40) & (data['gl'] <= 400)].copy()
    data['ts'] = pd.to_datetime(data['ts'])
    data['date'] = data['ts'].dt.date

    for patient, p_data in data.groupby('id'):
        p_data = p_data.sort_values('ts')
        
        current_start = p_data['date'].min()
        max_date = p_data['date'].max()
        
        # El bucle avanza en saltos de 14 días (7 para X + 7 para Y)
        # o 7 días si quieres que la 'Y' de una secuencia sea la 'X' de la siguiente
        while current_start + pd.Timedelta(days=days_week * 2) <= max_date:
            current_end = current_start + pd.Timedelta(days=days_week)
            next_end = current_end + pd.Timedelta(days=days_week)
            
            # Segmentación de ventanas
            mask_x = (p_data['date'] >= current_start) & (p_data['date'] < current_end)
            mask_y = (p_data['date'] >= current_end) & (p_data['date'] < next_end)
            
            week_x = p_data[mask_x]
            week_y = p_data[mask_y]
            
            # Verificación de calidad (Time Worn)
            if len(week_x) >= (expected_readings * 0.7) and len(week_y) >= (expected_readings * 0.7):
                
                # Cálculo de la etiqueta Y
                # Usamos la lógica de duraciones del código anterior si prefieres
                y_label = Event(week_y['gl'].tolist(), glucose_threshold)
                
                sequences.append({
                    'patient': patient,
                    'start_x': week_x['ts'].min(),
                    'end_x': week_x['ts'].max(),
                    'start_y': week_y['ts'].min(),
                    'Y': y_label,
                    'X': week_x['gl'].tolist(),
                    'L': len(week_x)
                })
            
            # --- EL CAMBIO CLAVE PARA VENTANA FIJA ---
            # Saltamos la semana completa para que no haya solapamiento
            current_start = current_end 
            
    return sequences

In [ ]:
New_Sequences_Fixed(Replace)

In [ ]:
from src.utils import get_labelled_sequences
# minimum percentage of time that the patient must have worn the device over a given week
time_worn_threshold = 0.7
# glucose threshold below which we detect the onset of hypoglycemia, in mg/dL
glucose_threshold = 54
# minimum length of a hypoglycemic event, in minutes
event_duration_threshold = 15
#determine the sequences

sequences = get_labelled_sequences(
    data=Replace,
    time_worn_threshold=time_worn_threshold,
    glucose_threshold=glucose_threshold,
    event_duration_threshold=event_duration_threshold,
)
print(len(sequences))

6718


In [ ]:
from src.utils import get_labelled_sequences
# minimum percentage of time that the patient must have worn the device over a given week
time_worn_threshold = 0.7
# glucose threshold below which we detect the onset of hypoglycemia, in mg/dL
glucose_threshold = 54
# minimum length of a hypoglycemic event, in minutes
event_duration_threshold = 15
#determine the sequences

sequences = get_labelled_sequences(
    data=AIDE,
    time_worn_threshold=time_worn_threshold,
    glucose_threshold=glucose_threshold,
    event_duration_threshold=event_duration_threshold,
)
print(len(sequences))

570


In [ ]:
from src.utils import get_labelled_sequences
# minimum percentage of time that the patient must have worn the device over a given week
time_worn_threshold = 0.7
# glucose threshold below which we detect the onset of hypoglycemia, in mg/dL
glucose_threshold = 54
# minimum length of a hypoglycemic event, in minutes
event_duration_threshold = 15
#determine the sequences

sequences = get_labelled_sequences(
    data=Shanghai,
    time_worn_threshold=time_worn_threshold,
    glucose_threshold=glucose_threshold,
    event_duration_threshold=event_duration_threshold,
)
print(len(sequences))

5
